# 🪐 FFI Exoplanet Detection Pipeline v7.0 — Physics-First

**Architecture (non-negotiable):**
```
FFI → Photometry → Quality Mask → Sector Normalize → Stitch
 → Detrend (CBV + Adaptive Wotan)
 → TLS (Primary Detection — Multi-pass)
 → Feature Engineering (Physical + Consistency + Stellar + Shape)
 → Scientific Vetting (Hard Filters)
 → ML Classification (Optional Enhancement — Stacking)
 → Output + Diagnostics
```

**Key design decisions:**
- TLS is the PRIMARY detection engine — ML does NOT detect signals
- LSTM-AE removed from detection chain
- ML classifies TLS candidates using stacking (not voting)
- 6 hard scientific vetting filters including duration consistency
- Multi-pass TLS for multiple candidate detection
- Gaia-informed stellar constraints throughout

## FFI Pipeline v7 — Part 1: Photometry, Detrending, TLS Detection

In [34]:
import os, warnings, logging, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from typing import Optional, Dict, Tuple, List
import numpy as np
import scipy.stats as stats
from scipy.ndimage import gaussian_filter1d
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
matplotlib.rcParams.update({"figure.dpi": 130, "axes.grid": True, "grid.alpha": 0.25})
import lightkurve as lk
from astroquery.gaia import Gaia
from astroquery.mast import Catalogs
import astropy.units as u
from astropy.timeseries import BoxLeastSquares
from wotan import flatten, slide_clip
from transitleastsquares import transitleastsquares, cleaned_array, catalog_info
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import xgboost as xgb
from tqdm.auto import tqdm
from joblib import Memory
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("ExoPipelineV7")
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
CACHE_DIR = Path("./pipeline_cache"); CACHE_DIR.mkdir(exist_ok=True)
memory = Memory(location=str(CACHE_DIR), verbose=0)
R_JUP_R_SUN = 0.10271
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends,"mps") and torch.backends.mps.is_available() else "cpu"))
print(f"✅ Device: {DEVICE}")

✅ Device: mps


In [35]:
# CONFIGURATION

In [36]:
@dataclass
class PipelineConfig:
    tic_id: str = "TIC 260647166"
    cadence: str = "long"
    author: str = "SPOC"
    quality_bitmask: str = "hardest"
    sigma_clip: float = 5.0
    min_sector_cadences: int = 200
    # BLS gate
    bls_min_period: float = 0.5
    bls_max_period: float = 27.0
    bls_n_periods: int = 8000
    bls_sde_threshold: float = 4.5
    # TLS (PRIMARY detector)
    tls_sde_threshold: float = 7.0
    tls_oversampling: int = 5
    tls_duration_grid_step: float = 1.05
    tls_max_passes: int = 3  # multi-pass TLS
    # Wotan detrending
    wotan_method: str = "biweight"
    transit_mask_duration_h: float = 4.0
    # CNN
    cnn_phase_bins: int = 128
    cnn_channels: List[int] = field(default_factory=lambda: [32, 64, 128])
    cnn_latent_dim: int = 64
    # Vetting (hard filters)
    max_rp_rjup: float = 2.0
    centroid_threshold_px: float = 0.3
    secondary_eclipse_threshold: float = 0.5
    odd_even_sigma_threshold: float = 3.0
    odd_even_ratio_limits: Tuple = (0.5, 2.0)
    duration_consistency_tolerance: float = 0.5  # fraction
    # Ensemble (stacking, not voting)
    high_confidence_threshold: float = 0.80
    random_seed: int = 42
    n_jobs: int = -1

In [37]:
# PHASE 1: DATA ACQUISITION & PHOTOMETRY

In [38]:
def _fetch_tess_lc(tic_id, cfg):
    """Download TESS light curves (SPOC with QLP fallback)."""
    logger.info(f"Fetching LCs for {tic_id} ...")
    sr = lk.search_lightcurve(tic_id, mission="TESS", cadence=cfg.cadence, author=cfg.author)
    if not sr:
        sr = lk.search_lightcurve(tic_id, mission="TESS", cadence="long", author="QLP")
    if not sr:
        raise RuntimeError(f"No TESS data for {tic_id}")
    return sr.download_all(quality_bitmask=cfg.quality_bitmask, cache=True)

def _fetch_tpf_lazy(tic_id, cfg):
    """Lazy TPF download — only called for vetting after detection."""
    logger.info(f"Lazy-loading TPFs for {tic_id} ...")
    sr = lk.search_targetpixelfile(tic_id, mission="TESS", cadence=cfg.cadence, author=cfg.author)
    return sr.download_all(quality_bitmask=cfg.quality_bitmask, cache=True) if sr else None

@memory.cache
def _fetch_gaia(tic_id):
    """Gaia DR3 stellar parameters with TIC fallback."""
    tic_num = tic_id.replace("TIC ", "").strip()
    res = Catalogs.query_criteria(catalog="TIC", ID=tic_num, objType="STAR")
    def _sf(v, fb):
        try:
            v2 = float(v)
            return v2 if np.isfinite(v2) else fb
        except: return fb
    try:
        ra, dec = float(res['ra'][0]), float(res['dec'][0])
        adql = (f"SELECT s.source_id, p.radius_gspphot, p.teff_gspphot, s.ruwe "
                f"FROM gaiadr3.astrophysical_parameters AS p "
                f"JOIN gaiadr3.gaia_source AS s ON p.source_id=s.source_id "
                f"WHERE 1=CONTAINS(POINT('ICRS',s.ra,s.dec),CIRCLE('ICRS',{ra},{dec},0.0003))")
        row = Gaia.launch_job(adql).get_results()[0]
        return {"radius_sun": _sf(row['radius_gspphot'], 1.0), "teff_k": _sf(row['teff_gspphot'], 5778.0),
                "ruwe": _sf(row.get('ruwe', 1.0), 1.0), "source": "GaiaDR3"}
    except:
        return {"radius_sun": _sf(res['rad'][0], 1.0) if res['rad'][0] else 1.0,
                "teff_k": _sf(res['Teff'][0], 5778.0) if res['Teff'][0] else 5778.0,
                "ruwe": np.nan, "source": "TIC"}

def ingest_parallel(cfg):
    """Parallel TESS + Gaia ingestion."""
    with ThreadPoolExecutor(max_workers=2) as ex:
        fut_lc = ex.submit(_fetch_tess_lc, cfg.tic_id, cfg)
        fut_g = ex.submit(_fetch_gaia, cfg.tic_id)
        return fut_lc.result(), fut_g.result()

In [39]:
# PHASE 2: PREPROCESSING (Quality + Per-sector Normalize + Stitch)

In [40]:
def preprocess_and_stitch(lc_coll, cfg):
    """Quality mask → sigma clip → per-sector normalize → CBV correct → stitch."""
    clean = []
    for lc in lc_coll:
        if len(lc) < cfg.min_sector_cadences:
            continue
        # Quality flag masking
        lc = lc[(lc.quality & (32 | 128 | 2048)) == 0]
        lc = lc.remove_outliers(sigma=cfg.sigma_clip, maxiters=5)
        lc = lc[np.isfinite(lc.flux.value)]
        if len(lc) < 100:
            continue
        # Per-sector normalization: flux / median(flux)
        clean.append(lc.normalize())

    if not clean:
        raise RuntimeError("No sectors survived quality filtering!")
    logger.info(f"{len(clean)}/{len(lc_coll)} sectors passed quality gate")

    # CBV correction (systematics removal)
    corrected = []
    for lc in clean:
        try:
            corr = lk.correctors.CBVCorrector(lc)
            corrected.append(corr.correct(cbv_type="SingleScale", alpha=1e-4).normalize())
        except:
            corrected.append(lc)

    # Extract CROWDSAP from FITS metadata
    crowdsap_vals = []
    for lc in corrected:
        try:
            cs = getattr(lc, 'meta', {}).get('CROWDSAP', None)
            if cs is not None:
                crowdsap_vals.append(float(cs))
        except: pass
    crowding = np.nanmedian(crowdsap_vals) if crowdsap_vals else 1.0
    if not np.isfinite(crowding): crowding = 1.0

    # Stitch + sort by time
    stitched = lk.LightCurveCollection(corrected).stitch()
    idx = np.argsort(stitched.time.value)
    stitched = stitched[idx]
    logger.info(f"Stitched: {len(stitched)} cadences, crowding={crowding:.4f}")
    return stitched.normalize(), crowding

In [41]:
# PHASE 3: DETRENDING (CBV already done + Wotan adaptive window)

In [42]:
def build_transit_mask(time, period, t0, duration_h=4.0):
    dur_days = duration_h / 24.0
    phase = ((time - t0) % period) / period
    phase = np.where(phase > 0.5, phase - 1.0, phase)
    return np.abs(phase) < (dur_days / period)

def bls_prescan(time, flux, cfg):
    """Fast BLS pre-scan to gate expensive TLS."""
    logger.info("BLS pre-scan ...")
    bls = BoxLeastSquares(time * u.day, flux)
    periods = np.linspace(cfg.bls_min_period, cfg.bls_max_period, cfg.bls_n_periods)
    result = bls.power(periods * u.day, [0.01, 0.02, 0.05, 0.10] * u.day)
    best_idx = int(np.argmax(result.power))
    best_period = float(result.period[best_idx].value)
    best_t0 = float(result.transit_time[best_idx].value)
    depth = float(result.depth[best_idx])
    sde = float((result.power[best_idx] - np.median(result.power)) / (np.std(result.power) + 1e-12))
    logger.info(f"BLS: P={best_period:.4f}d  SDE={sde:.2f}  depth={depth:.5f}")
    return best_period, best_t0, depth, sde

def detrend_wotan(time, flux, transit_mask, cfg, expected_duration_h=None):
    """Wotan biweight with adaptive window ≈ 1-3× expected transit duration."""
    # Adaptive window: if we know expected duration, use 3× that; else default 0.5d
    if expected_duration_h and expected_duration_h > 0:
        window = min(max(3.0 * expected_duration_h / 24.0, 0.3), 1.0)
    else:
        window = 0.5
    logger.info(f"Wotan detrending (window={window:.3f}d) ...")
    flat_flux, trend = flatten(time, flux, window_length=window, method=cfg.wotan_method,
        mask=transit_mask, return_trend=True, break_tolerance=0.5, robust=True)
    flat_flux = slide_clip(time, flat_flux, window_length=1.0,
        low=cfg.sigma_clip, high=cfg.sigma_clip, method="mad", center="median")
    return flat_flux, trend

In [43]:
# PHASE 4: TLS — PRIMARY DETECTION ENGINE (Multi-pass)

In [ ]:
# --- HIGH PERFORMANCE PRODUCTION CONFIG ---
cfg = PipelineConfig(tic_id="TIC 307210830")
cfg.tls_oversampling = 3           # Scientific standard (high precision)
cfg.tls_duration_grid_step = 1.05  # Fine grid for accurate transit shapes
cfg.bls_max_period = 20.0          # Sufficient for most FFI searches
cfg.tls_max_passes = 2             # Find primary and secondary candidates

import os
# Use all available CPU cores (Cuda/GPU doesn't speed up TLS, but more CPU cores do)
n_cores = os.cpu_count() or 4
print(f"⚙️ Using {n_cores} CPU cores for High-Precision TLS...")

def run_tls_production(time, flux, stellar, cfg):
    """High-precision TLS without performance degradation."""
    try:
        ab, R_s, M_s = catalog_info(TIC_ID=int(cfg.tic_id.replace("TIC ", "")))
    except:
        ab, R_s, M_s = [0.4, 0.3], stellar.get("radius_sun", 1.0), 1.0
    
    t_c, f_c = cleaned_array(time, flux)
    
    # We do NOT bin data here to maintain maximum shape precision
    tls = transitleastsquares(t_c, f_c)
    
    print(f"🔍 Searching with {cfg.tls_oversampling}x oversampling (High Precision)...")
    
    result = tls.power(
        R_star=float(R_s), 
        M_star=float(M_s), 
        u=list(ab),
        period_min=0.5, 
        period_max=cfg.bls_max_period, 
        oversampling_factor=cfg.tls_oversampling, 
        duration_grid_step=cfg.tls_duration_grid_step,
        use_threads=n_cores, # Uses all cores
        show_progress_bar=True
    )
    return result

# --- EXECUTE ---
try:
    # 1. Ingest & Detrend (Full v7 Logic)
    lc_coll, stellar = ingest_parallel(cfg)
    lc_s, crowdsap = preprocess_and_stitch(lc_coll, cfg)
    
    # Detrend using the adaptive Wotan window
    t_mask = build_transit_mask(lc_s.time.value, 1.0, 0.0, 4.0) # Dummy mask for initial detrend
    f_flat, trend = detrend_wotan(lc_s.time.value, lc_s.flux.value, t_mask, cfg)
    
    # 2. Run High-Precision TLS
    tls_result = run_tls_production(lc_s.time.value[np.isfinite(f_flat)], f_flat[np.isfinite(f_flat)], stellar, cfg)
    
    # 3. Features & Vetting
    feats, profile = extract_features(lc_s.time.value, f_flat, tls_result, stellar, 10.0, crowdsap)
    cnn_vec = extract_cnn_vector(profile, cfg)
    tpf_coll = _fetch_tpf_lazy(cfg.tic_id, cfg)
    vet = vet_candidate(feats, tpf_coll, tls_result, cfg)
    pred = classify_candidate(feats, vet, cnn_vec, cfg)

    # 4. Final Result
    print(f"\n✅ FULL PERFORMANCE RUN COMPLETE")
    print(f"Verdict: {pred.verdict} ({pred.confidence} confidence)")
    print(f"Period:  {feats['period']:.5f} days")
    print(f"Rp:      {feats['rp_rjup']:.4f} R_Jup")
    
    # Diagnostic Plot
    plot_diagnostics(lc_s.time.value, lc_s.flux.value, lc_s.time.value, f_flat, trend, 
                     {"tls_result": tls_result, "features": feats, "vetting": vet, "prediction": pred, "profile": profile}, cfg)

except Exception as e:
    print(f"❌ Error: {e}")


def multi_pass_tls(time, flux, stellar, cfg):
    """Multi-pass High-Precision TLS."""
    candidates = []
    t_work, f_work = time.copy(), flux.copy()
    
    for pass_num in range(cfg.tls_max_passes):
        logger.info(f"--- Starting High-Precision TLS Pass {pass_num+1} ---")
        
        try:
            # Change: Call 'run_tls_production' instead of 'run_tls'
            tls_res = run_tls_production(t_work, f_work, stellar, cfg)
            
            # SDE check to see if signal is strong enough to keep going
            if tls_res.SDE < cfg.tls_sde_threshold:
                logger.info(f"SDE={tls_res.SDE:.2f} too low → Stopping search.")
                break
            
            candidates.append(tls_res)
            
            # Mask the found transit so we can find the next one
            mask = build_transit_mask(t_work, tls_res.period, tls_res.T0, tls_res.duration * 24)
            f_work = np.where(mask, np.nanmedian(f_work), f_work)
            
        except Exception as e:
            logger.error(f"Error in TLS pass {pass_num+1}: {e}")
            break
            
    return candidates


2026-04-28 12:48:39,797 | INFO | Fetching LCs for TIC 307210830 ...


⚙️ Using 8 CPU cores for High-Precision TLS...


2026-04-28 12:48:39,895 | WARNING | Warning: 25% (315/1245) of the cadences will be ignored due to the quality mask (quality_bitmask=65535).
2026-04-28 12:48:39,910 | WARNING | Warning: 25% (315/1245) of the cadences will be ignored due to the quality mask (quality_bitmask=4096).
14% (162/1196) of the cadences will be ignored due to the quality mask (quality_bitmask=65535).
2026-04-28 12:48:39,936 | INFO | 14% (162/1196) of the cadences will be ignored due to the quality mask (quality_bitmask=65535).
14% (162/1196) of the cadences will be ignored due to the quality mask (quality_bitmask=4096).
2026-04-28 12:48:39,938 | INFO | 14% (162/1196) of the cadences will be ignored due to the quality mask (quality_bitmask=4096).
2026-04-28 12:48:39,970 | WARNING | Warning: 30% (295/968) of the cadences will be ignored due to the quality mask (quality_bitmask=65535).
2026-04-28 12:48:39,973 | WARNING | Warning: 30% (295/968) of the cadences will be ignored due to the quality mask (quality_bitmask

🔍 Searching with 3x oversampling (High Precision)...
Transit Least Squares TLS 1.32 (5 Apr 2024)
Creating model cache for 78 durations
Searching 163752 data points, 909132 periods from 0.5 to 20.0 days
Using all 8 CPU threads


## FFI Pipeline v7 — Part 2: Features, Vetting, ML, Runner

In [ ]:
# PHASE 5: FEATURE ENGINEERING (Physical + Consistency + Stellar + Shape)

In [ ]:
def phase_fold_profile(time, flux, period, t0, n_bins=128):
    phase = ((time - t0) % period) / period
    phase = np.where(phase > 0.5, phase - 1.0, phase)
    idx = np.argsort(phase); ph_s, fl_s = phase[idx], flux[idx]
    bins = np.linspace(-0.5, 0.5, n_bins + 1)
    binned = np.ones(n_bins, dtype=np.float64)
    for i in range(n_bins):
        m = (ph_s >= bins[i]) & (ph_s < bins[i+1])
        if m.sum() > 0: binned[i] = float(np.median(fl_s[m]))
    rng = binned.max() - binned.min() + 1e-8
    return ((binned - binned.min()) / rng).astype(np.float32)

class CNN1DEncoder(nn.Module):
    def __init__(self, input_len=128, channels=(32,64,128), latent_dim=64):
        super().__init__()
        layers, in_ch = [], 1
        for out_ch in channels:
            layers += [nn.Conv1d(in_ch, out_ch, 5, padding=2), nn.BatchNorm1d(out_ch), nn.GELU(), nn.MaxPool1d(2), nn.Dropout(0.2)]
            in_ch = out_ch
        self.conv = nn.Sequential(*layers)
        with torch.no_grad(): flat_size = self.conv(torch.zeros(1,1,input_len)).view(1,-1).shape[1]
        self.fc = nn.Sequential(nn.Linear(flat_size, 256), nn.GELU(), nn.Dropout(0.3), nn.Linear(256, latent_dim))
    def forward(self, x): return self.fc(self.conv(x).view(x.size(0), -1))

def extract_cnn_vector(profile, cfg, model=None):
    cnn = CNN1DEncoder(cfg.cnn_phase_bins, tuple(cfg.cnn_channels), cfg.cnn_latent_dim).to(DEVICE)
    if model is not None: cnn.load_state_dict(model.state_dict())
    cnn.eval()
    with torch.no_grad():
        return cnn(torch.tensor(profile[None, None, :]).to(DEVICE)).cpu().numpy().squeeze()

def _sf(x, fb=0.0):
    try:
        v = float(np.atleast_1d(x).flat[0])
        return v if np.isfinite(v) else fb
    except: return fb

def _odd_even(time, flux, period, t0):
    phase = ((time - t0) % period) / period
    phase = np.where(phase > 0.5, phase - 1.0, phase)
    in_tr = np.abs(phase) < 0.02
    if in_tr.sum() < 5: return 0.0, 0.0, 1.0, 0.0
    n_tr = np.round((time[in_tr] - t0) / period).astype(int)
    odd_d, even_d = [], []
    for n in np.unique(n_tr):
        m = n_tr == n; d = 1.0 - float(np.median(flux[in_tr][m]))
        (odd_d if n % 2 else even_d).append(d)
    om = float(np.mean(odd_d)) if odd_d else 0.0
    em = float(np.mean(even_d)) if even_d else 0.0
    # Statistical sigma difference
    if odd_d and even_d:
        sigma = abs(om - em) / (np.sqrt(np.var(odd_d)/len(odd_d) + np.var(even_d)/len(even_d)) + 1e-10)
    else: sigma = 0.0
    return om, em, om / (em + 1e-8), sigma

def _secondary_depth(time, flux, period, t0):
    phase = ((time - t0) % period) / period
    mask = np.abs(phase - 0.5) < 0.02
    return float(1.0 - np.median(flux[mask])) if mask.sum() >= 3 else 0.0

def _transit_snr(flux, in_mask):
    if in_mask.sum() == 0: return 0.0
    return (1.0 - float(np.median(flux[in_mask]))) / (float(np.std(flux[~in_mask])) + 1e-10)

def _expected_duration_h(period_d, r_star_sun, m_star_sun=1.0):
    """Kepler's 3rd law estimate of transit duration (hours)."""
    a_au = (m_star_sun * (period_d / 365.25)**2)**(1/3)
    a_rsun = a_au * 215.0
    dur_d = (r_star_sun * period_d) / (np.pi * a_rsun + 1e-10)
    return dur_d * 24.0

def extract_features(time, flux, tls_res, stellar, bls_sde, crowdsap):
    """Extract physical + consistency + stellar-informed features."""
    if tls_res is None:
        return {k: 0.0 for k in ["period","depth","duration_h","sde","snr","rp_rjup",
            "odd_depth","even_depth","odd_even_ratio","odd_even_sigma","secondary_depth",
            "n_transits","transit_quality","rstar","teff","ruwe","bls_sde",
            "duration_expected_h","duration_ratio","in_transit_count"]}, None

    R_star = stellar.get("radius_sun", 1.0)
    depth = _sf(tls_res.depth)
    true_depth = depth / max(crowdsap, 0.01)
    rp = R_star * np.sqrt(max(true_depth, 0.0)) / R_JUP_R_SUN
    period = _sf(tls_res.period)
    dur_h = _sf(tls_res.duration) * 24
    t_mask = build_transit_mask(time, period, _sf(tls_res.T0), dur_h)
    snr = _transit_snr(flux, t_mask)
    odd_d, even_d, oe_ratio, oe_sigma = _odd_even(time, flux, period, _sf(tls_res.T0))
    sec_d = _secondary_depth(time, flux, period, _sf(tls_res.T0))
    n_obs = len(tls_res.transit_times) if hasattr(tls_res, "transit_times") else 0
    baseline = float(time[-1] - time[0])
    n_exp = baseline / (period + 1e-8)
    exp_dur = _expected_duration_h(period, R_star)
    dur_ratio = dur_h / (exp_dur + 1e-8)
    # Phase-folded profile for CNN
    profile = phase_fold_profile(time, flux, period, _sf(tls_res.T0))
    feats = {
        "period": period, "depth": true_depth, "duration_h": dur_h,
        "sde": _sf(tls_res.SDE), "snr": snr, "rp_rjup": rp,
        "odd_depth": odd_d, "even_depth": even_d, "odd_even_ratio": oe_ratio,
        "odd_even_sigma": oe_sigma, "secondary_depth": sec_d,
        "n_transits": float(n_obs), "transit_quality": n_obs / (n_exp + 1e-8),
        "rstar": R_star, "teff": stellar.get("teff_k", 5778.0),
        "ruwe": stellar.get("ruwe", 1.0), "bls_sde": bls_sde,
        "duration_expected_h": exp_dur, "duration_ratio": dur_ratio,
        "in_transit_count": float(t_mask.sum()),
    }
    return feats, profile

In [ ]:
# PHASE 6: SCIENTIFIC VETTING (Hard Filters)

In [ ]:
@dataclass
class VettingResult:
    passed_centroid: bool = True
    passed_radius: bool = True
    passed_secondary: bool = True
    passed_odd_even: bool = True
    passed_duration: bool = True
    passed_transit_cnt: bool = True
    centroid_shift_px: float = 0.0
    fp_score: float = 0.0
    fp_reasons: List = field(default_factory=list)

def _centroid_shift(tpf_coll, tls_res, cfg):
    if tpf_coll is None or tls_res is None: return 0.0
    shifts = []
    for tpf in tpf_coll:
        try:
            t = tpf.time.value
            in_tr = build_transit_mask(t, tls_res.period, tls_res.T0, tls_res.duration * 24)
            if in_tr.sum() < 3 or (~in_tr).sum() < 3: continue
            f = tpf.flux.value
            R, C = np.meshgrid(np.arange(f.shape[1]), np.arange(f.shape[2]), indexing="ij")
            def com(frames):
                fr = np.clip(np.nanmean(frames, axis=0), 0, None); s = fr.sum() + 1e-12
                return (fr*C).sum()/s, (fr*R).sum()/s
            ci, co = com(f[in_tr]), com(f[~in_tr])
            shifts.append(np.hypot(ci[0]-co[0], ci[1]-co[1]))
        except: continue
    return float(np.median(shifts)) if shifts else 0.0

def vet_candidate(feats, tpf_coll, tls_res, cfg):
    """Scientific vetting with hard filters."""
    v = VettingResult(); fp = 0.0
    # 1. Centroid (difference imaging proxy)
    shift = _centroid_shift(tpf_coll, tls_res, cfg)
    v.centroid_shift_px = shift
    if shift > cfg.centroid_threshold_px:
        v.passed_centroid = False; v.fp_reasons.append(f"Centroid {shift:.3f}px"); fp += 0.40
    # 2. Radius sanity (soft constraint)
    rp = feats["rp_rjup"]
    if rp > cfg.max_rp_rjup:
        v.passed_radius = False; v.fp_reasons.append(f"Rp={rp:.2f} RJup"); fp += 0.35
    # 3. Secondary eclipse (statistical)
    prim = feats["depth"] + 1e-10
    if feats["secondary_depth"] / prim > cfg.secondary_eclipse_threshold:
        v.passed_secondary = False; v.fp_reasons.append("Secondary eclipse detected"); fp += 0.30
    # 4. Odd-even test (statistical, not visual)
    if feats["odd_even_sigma"] > cfg.odd_even_sigma_threshold:
        v.passed_odd_even = False; v.fp_reasons.append(f"Odd/even {feats['odd_even_sigma']:.1f}σ"); fp += 0.25
    # 5. Duration consistency with orbital expectation
    if abs(feats["duration_ratio"] - 1.0) > cfg.duration_consistency_tolerance:
        v.passed_duration = False; v.fp_reasons.append(f"Duration ratio={feats['duration_ratio']:.2f}"); fp += 0.15
    # 6. Transit count
    if feats["n_transits"] < 2:
        v.passed_transit_cnt = False; v.fp_reasons.append("n_transits < 2"); fp += 0.10
    v.fp_score = float(min(fp, 1.0))
    return v

In [ ]:
# PHASE 7: ML CLASSIFICATION (Classify TLS candidates, NOT detect)

In [ ]:
FEAT_KEYS = ["period","depth","duration_h","sde","snr","rp_rjup","odd_depth","even_depth",
    "odd_even_ratio","odd_even_sigma","secondary_depth","n_transits","transit_quality",
    "rstar","teff","ruwe","bls_sde","duration_expected_h","duration_ratio","in_transit_count"]

def assemble_features(feats, vet, cnn_vec):
    phys = np.array([feats[k] for k in FEAT_KEYS], dtype=np.float32)
    vet_arr = np.array([vet.centroid_shift_px, vet.fp_score, float(vet.passed_centroid),
        float(vet.passed_radius), float(vet.passed_secondary), float(vet.passed_odd_even),
        float(vet.passed_duration)], dtype=np.float32)
    phys_vec = np.concatenate([phys, vet_arr])
    return np.concatenate([cnn_vec.astype(np.float32), phys_vec]), phys_vec

def build_stacking_classifier(cfg):
    """Stacking ensemble: CNN features feed XGB+RF → LogReg meta-learner."""
    base = [
        ("xgb", xgb.XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
            eval_metric="logloss", random_state=cfg.random_seed, tree_method="hist")),
        ("rf", RandomForestClassifier(n_estimators=300, class_weight="balanced",
            random_state=cfg.random_seed, n_jobs=cfg.n_jobs)),
    ]
    return StackingClassifier(estimators=base, final_estimator=LogisticRegression(), cv=5, n_jobs=cfg.n_jobs)

def heuristic_score(feats, vet):
    """Fallback when no trained classifier is available."""
    sde_norm = float(np.clip(feats["sde"] / 15.0, 0, 1))
    return float(sde_norm * (1.0 - vet.fp_score))

@dataclass
class Prediction:
    probability: float = 0.0
    verdict: str = "UNKNOWN"
    confidence: str = "LOW"
    explanation: str = ""
    method: str = "heuristic"

def classify_candidate(feats, vet, cnn_vec, cfg, trained_models=None):
    """Classify a TLS candidate. Uses trained stacking if available, else heuristic."""
    if trained_models is not None:
        full_v, phys_v = assemble_features(feats, vet, cnn_vec)
        sc = trained_models["scaler"].transform(phys_v.reshape(1, -1))
        prob = float(trained_models["model"].predict_proba(sc)[0, 1])
        method = "stacking"
    else:
        prob = heuristic_score(feats, vet)
        method = "heuristic"
    # Apply vetting hard overrides
    pred = Prediction(probability=prob, method=method)
    if not vet.passed_radius:
        pred.verdict, pred.confidence = "FALSE_POSITIVE", "HIGH"
        pred.explanation = f"Rp > {cfg.max_rp_rjup} RJup — likely EB"; return pred
    if not vet.passed_centroid:
        pred.verdict, pred.confidence = "FALSE_POSITIVE", "HIGH"
        pred.explanation = "Centroid shift — background EB"; return pred
    if not vet.passed_secondary:
        pred.verdict, pred.confidence = "FALSE_POSITIVE", "MEDIUM"
        pred.explanation = "Secondary eclipse — EB system"; return pred
    # Soft classification
    if prob >= cfg.high_confidence_threshold and vet.fp_score < 0.2:
        pred.verdict, pred.confidence = "PLANET_CANDIDATE", "HIGH"
        pred.explanation = f"p={prob:.3f}, all vetting passed"
    elif prob >= 0.5:
        pred.verdict, pred.confidence = "PLANET_CANDIDATE", "MEDIUM"
        pred.explanation = f"p={prob:.3f}, review recommended"
    elif prob >= 0.3:
        pred.verdict, pred.confidence = "WEAK_SIGNAL", "LOW"
        pred.explanation = f"p={prob:.3f}, marginal detection"
    else:
        pred.verdict, pred.confidence = "NON_DETECTION", "HIGH"
        pred.explanation = f"p={prob:.3f}, no planet signal"
    return pred

In [ ]:
# PHASE 8: FULL PIPELINE RUNNER

In [ ]:
def run_pipeline(tic_id, cfg=None, trained_models=None, plot=True):
    if cfg is None: cfg = PipelineConfig(tic_id=tic_id)
    else: cfg.tic_id = tic_id
    logger.info(f"{'='*50}\nPipeline v7: {tic_id}\n{'='*50}")
    # 1. Ingest
    lc_coll, stellar = ingest_parallel(cfg)
    # 2. Preprocess + stitch
    lc_s, crowdsap = preprocess_and_stitch(lc_coll, cfg)
    t_raw, f_raw = lc_s.time.value, lc_s.flux.value
    # 3. BLS gate
    b_p, b_t0, b_dep, b_sde = bls_prescan(t_raw, f_raw, cfg)
    if b_sde < cfg.bls_sde_threshold:
        logger.info(f"BLS SDE={b_sde:.2f} < gate → NO_SIGNAL")
        return {"tic_id": tic_id, "verdict": "NO_SIGNAL", "bls_sde": b_sde, "candidates": []}
    # 4. Detrend
    t_mask = build_transit_mask(t_raw, b_p, b_t0, cfg.transit_mask_duration_h)
    f_flat, trend = detrend_wotan(t_raw, f_raw, t_mask, cfg)
    valid = np.isfinite(f_flat)
    t_det, f_det = t_raw[valid], f_flat[valid]
    # 5. Multi-pass TLS
    candidates = multi_pass_tls(t_det, f_det, stellar, cfg)
    if not candidates:
        return {"tic_id": tic_id, "verdict": "NO_DETECTION", "bls_sde": b_sde, "candidates": []}
    # 6. Process each candidate
    results = []
    tpf_coll = _fetch_tpf_lazy(tic_id, cfg)
    for i, tls_res in enumerate(candidates):
        feats, profile = extract_features(t_det, f_det, tls_res, stellar, b_sde, crowdsap)
        if profile is None: continue
        cnn_vec = extract_cnn_vector(profile, cfg)
        vet = vet_candidate(feats, tpf_coll, tls_res, cfg)
        pred = classify_candidate(feats, vet, cnn_vec, cfg, trained_models)
        results.append({"candidate": i+1, "tls_result": tls_res, "features": feats,
            "profile": profile, "vetting": vet, "prediction": pred})
        logger.info(f"Candidate {i+1}: {pred.verdict} ({pred.confidence}) P={feats['period']:.4f}d")
    best = results[0] if results else None
    if plot and best:
        plot_diagnostics(t_raw, f_raw, t_det, f_det, trend, best, cfg)
    return {"tic_id": tic_id, "verdict": best["prediction"].verdict if best else "NO_DETECTION",
        "bls_sde": b_sde, "stellar": stellar, "crowdsap": crowdsap, "candidates": results,
        "n_sectors": len(lc_coll), "time_det": t_det, "flux_det": f_det}

In [ ]:
# DIAGNOSTIC PLOT

In [ ]:
def plot_diagnostics(t_raw, f_raw, t_det, f_det, trend, candidate, cfg, save_path=None):
    C = {"bg":"#0d1117","card":"#161b22","grid":"#21262d","txt":"#e6edf3",
         "dim":"#8b949e","blue":"#58a6ff","red":"#f85149","grn":"#3fb950","yel":"#d29922"}
    fig = plt.figure(figsize=(18, 16), facecolor=C["bg"])
    gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.42, wspace=0.3)
    tls_res = candidate["tls_result"]; pf = candidate["features"]
    vet = candidate["vetting"]; pred = candidate["prediction"]
    def style(ax, title):
        ax.set_facecolor(C["card"]); ax.set_title(title, color=C["txt"], fontsize=10.5, pad=7)
        ax.tick_params(colors=C["dim"], labelsize=8.5)
        for s in ax.spines.values(): s.set_color(C["grid"])
        ax.xaxis.label.set_color(C["dim"]); ax.yaxis.label.set_color(C["dim"])
        ax.grid(color=C["grid"], linewidth=0.5)
    # 1. Raw LC
    ax1 = fig.add_subplot(gs[0, :]); ax1.scatter(t_raw, f_raw, s=0.5, alpha=0.55, color=C["blue"], rasterized=True)
    if tls_res:
        for t0 in tls_res.transit_times: ax1.axvline(t0, color=C["grn"], alpha=0.3, lw=0.8)
    style(ax1, f"Raw Light Curve — {cfg.tic_id}"); ax1.set_xlabel("BJD"); ax1.set_ylabel("Flux")
    # 2. TLS periodogram
    ax2 = fig.add_subplot(gs[1, 0])
    if tls_res:
        ax2.plot(tls_res.periods, tls_res.power, color=C["blue"], lw=0.65)
        ax2.axvline(tls_res.period, color=C["grn"], lw=1.5, label=f"P={tls_res.period:.4f}d")
        ax2.axhline(cfg.tls_sde_threshold, color=C["red"], lw=1, ls="--")
    style(ax2, f"TLS Periodogram (SDE={pf['sde']:.2f})"); ax2.set_xlabel("Period (d)"); ax2.set_ylabel("SDE")
    # 3. Phase-folded
    ax3 = fig.add_subplot(gs[1, 1])
    ph_bins = np.linspace(-0.5, 0.5, cfg.cnn_phase_bins)
    ax3.plot(ph_bins, candidate["profile"], color=C["blue"], lw=1.4)
    style(ax3, "Phase-Folded Transit"); ax3.set_xlabel("Phase"); ax3.set_ylabel("Norm. Flux")
    # 4. Summary
    ax4 = fig.add_subplot(gs[2, :]); ax4.set_facecolor(C["card"]); ax4.axis("off")
    vc = C["grn"] if "PLANET" in pred.verdict else C["red"]
    rows = [("VERDICT", pred.verdict, vc), ("Confidence", pred.confidence, C["txt"]),
        ("P(planet)", f"{pred.probability:.4f} [{pred.method}]", C["txt"]),
        ("Period", f"{pf['period']:.5f} d", C["txt"]), ("Depth", f"{pf['depth']:.5f}", C["txt"]),
        ("Rp", f"{pf['rp_rjup']:.3f} RJup", C["txt"]), ("Duration", f"{pf['duration_h']:.2f}h (exp: {pf['duration_expected_h']:.2f}h)", C["txt"]),
        ("Centroid", f"{'PASS' if vet.passed_centroid else 'FAIL'} ({vet.centroid_shift_px:.3f}px)", C["grn"] if vet.passed_centroid else C["red"]),
        ("Odd/Even", f"{'PASS' if vet.passed_odd_even else 'FAIL'} ({pf['odd_even_sigma']:.1f}σ)", C["grn"] if vet.passed_odd_even else C["red"]),
        ("Secondary", f"{'PASS' if vet.passed_secondary else 'FAIL'}", C["grn"] if vet.passed_secondary else C["red"]),
        ("Duration Check", f"{'PASS' if vet.passed_duration else 'FAIL'} (ratio={pf['duration_ratio']:.2f})", C["grn"] if vet.passed_duration else C["red"]),
        ("FP Score", f"{vet.fp_score:.3f}", C["grn"] if vet.fp_score < 0.2 else C["red"])]
    for i, (lbl, val, col) in enumerate(rows):
        y = 0.92 - i * 0.07
        ax4.text(0.03, y, lbl, color=C["dim"], fontsize=9, transform=ax4.transAxes, va="top")
        ax4.text(0.35, y, val, color=col, fontsize=9, transform=ax4.transAxes, va="top", fontweight="bold")
    fig.suptitle(f"FFI Pipeline v7 — {cfg.tic_id}", color=C["txt"], fontsize=13, fontweight="bold", y=0.997)
    out = save_path or f"diagnostic_{cfg.tic_id.replace(' ','_')}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=C["bg"]); plt.close()
    logger.info(f"Plot → {out}")

In [ ]:
# BATCH RUNNER

In [ ]:
def batch_search(tics, trained_models=None, plot=False):
    records = []
    for tic in tics:
        try:
            r = run_pipeline(tic, trained_models=trained_models, plot=plot)
            best = r["candidates"][0] if r["candidates"] else None
            pf = best["features"] if best else {}
            records.append({"tic_id": tic, "verdict": r["verdict"],
                "p_planet": round(best["prediction"].probability, 4) if best else 0,
                "bls_sde": round(r["bls_sde"], 3),
                "period_d": round(pf.get("period", 0), 5),
                "rp_rjup": round(pf.get("rp_rjup", 0), 3),
                "n_candidates": len(r["candidates"]), "n_sectors": r.get("n_sectors", 0)})
        except Exception as e:
            logger.warning(f"{tic} failed: {e}")
            records.append({"tic_id": tic, "verdict": "ERROR", "p_planet": 0})
    df = pd.DataFrame(records).sort_values("p_planet", ascending=False)
    df.to_csv("batch_results_v7.csv", index=False)
    print(df.to_string(index=False))
    return df

if __name__ == "__main__":
    cfg = PipelineConfig()
    print("Pipeline v7 ready. Call run_pipeline('TIC XXXXX') or batch_search([...]).")

## 🚀 Run Pipeline

In [ ]:
cfg = PipelineConfig()
print(f"Device: {DEVICE}")

# Single target
result = run_pipeline("TIC 307210830", cfg=cfg, plot=True)
print(f"Verdict: {result['verdict']}")
if result["candidates"]:
    c = result["candidates"][0]
    print(f"P={c['features']['period']:.5f}d  Rp={c['features']['rp_rjup']:.3f} RJup")

In [ ]:
# Batch search
#df = batch_search(["TIC 307210830", "TIC 260647166", "TIC 402026209"], plot=False)
#df